In [ ]:
# Install Dependencies
!pip install torch torchvision transformers paddleocr
# Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
from transformers import SwinTransformerModel, SwinTransformerBlock
from paddleocr import PaddleOCR
from kornia.filters import MotionBlur

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Swin UNet Generator with Updated Decoder
class SwinUNet(nn.Module):
    def __init__(self, img_size=(128, 256), patch_size=4, in_chans=3, out_chans=3):
        super(SwinUNet, self).__init__()
        self.swin = SwinTransformerModel.from_pretrained(
            "microsoft/swin-tiny-patch4-window7-224",
            image_size=img_size,
            patch_size=patch_size,
            in_channels=in_chans
        )
        self.enc_dims = [96, 192, 384, 768]
        
        # Patch Expanding Layer
        class PatchExpand(nn.Module):
            def __init__(self, in_dim, out_dim, scale_factor=2):
                super(PatchExpand, self).__init__()
                self.scale_factor = scale_factor
                self.linear = nn.Linear(in_dim * scale_factor**2, out_dim)
            
            def forward(self, x):
                b, c, h, w = x.shape
                x = x.permute(0, 2, 3, 1).reshape(b, h * w, c)
                x = x.view(b, h, w, c).permute(0, 3, 1, 2)
                x = F.pixel_shuffle(x, self.scale_factor)
                x = x.permute(0, 2, 3, 1).reshape(b, h * self.scale_factor * w * self.scale_factor, c * self.scale_factor**2)
                x = self.linear(x)
                x = x.view(b, h * self.scale_factor, w * self.scale_factor, -1).permute(0, 3, 1, 2)
                return x
        
        # Decoder with Swin Transformer Blocks and Patch Expanding
        self.decoder = nn.ModuleList([
            PatchExpand(self.enc_dims[3], self.enc_dims[2]),
            SwinTransformerBlock(dim=self.enc_dims[2] * 2, num_heads=12, window_size=4),
            PatchExpand(self.enc_dims[2], self.enc_dims[1]),
            SwinTransformerBlock(dim=self.enc_dims[1] * 2, num_heads=6, window_size=4),
            PatchExpand(self.enc_dims[1], self.enc_dims[0]),
            nn.Conv2d(self.enc_dims[0] * 2, self.enc_dims[0], kernel_size=3, padding=1),
            nn.Conv2d(self.enc_dims[0], out_chans, kernel_size=3, padding=1)
        ])
        self.scale_convs = nn.ModuleList([
            nn.Conv2d(self.enc_dims[2] * 2, out_chans, kernel_size=3, padding=1),
            nn.Conv2d(self.enc_dims[1] * 2, out_chans, kernel_size=3, padding=1)
        ])
        
        # Text Reconstruction Module (TRM) for PaddleOCR
        self.trm = nn.Sequential(
            nn.Conv2d(self.enc_dims[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 256)  # Text feature vector
        )
    
    def forward(self, x):
        enc_features = self.swin(pixel_values=x).hidden_states[1:]
        x = enc_features[-1]
        outputs = []
        for i in range(0, 6, 2):
            x = self.decoder[i](x)
            skip_idx = len(enc_features) - 2 - (i // 2)
            if skip_idx >= 0:
                x = torch.cat([x, enc_features[skip_idx]], dim=1)
            x = self.decoder[i + 1](x)
            if i < 4:  # Save intermediate outputs
                outputs.append(torch.tanh(self.scale_convs[i // 2](x)))
        img_output = torch.tanh(self.decoder[-1](x))
        outputs.append(img_output)
        text_output = self.trm(img_output)
        return outputs, text_output

In [ ]:
# Discriminator with Partition Discriminator Module
class Discriminator(nn.Module):
    def __init__(self, img_size=(128, 256), in_chans=3, patch_size=16):
        super(Discriminator, self).__init__()
        self.global_disc = nn.Sequential(
            nn.Conv2d(in_chans, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0)
        )
        
        self.patch_size = patch_size
        self.patch_disc = nn.Sequential(
            nn.Conv2d(in_chans, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1)
        )
    
    def forward(self, x):
        global_out = self.global_disc(x)
        b, c, h, w = x.shape
        patches = x.unfold(2, self.patch_size, self.patch_size // 4).unfold(3, self.patch_size, self.patch_size// 4)
        patches = patches.contiguous().view(b, c, -1, self.patch_size, self.patch_size)
        patch_out = []
        for i in range(patches.size(2), dim=2):
            patch = patches[:, :, i], :, :]
            patch_out.append(self.patch_disc(patch))
        patch_out = torch.stack(patch_out, dim=2)
        return global_out, patch_out

In [ ]:
# VGG Perceptual Loss
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True).features
        self.vgg = nn.Sequential(
            *list(vgg.children())[:16]
        ).eval()
        for param in self.vgg.parameters():
            param.requires_grad = False
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    def forward(self, x, y):
        x = self.normalize(x)
        y = self.normalize(y)
        x_vgg = self.vgg(x)
        y_vgg = self.vgg(y)
        return F.mse_loss(x_vgg, y_vgg)

In [ ]:
# Custom Dataset with Flip Correction
class LicensePlateDataset(Dataset):
    def __init__(self, blur_dir, sharp_dir, transform=None):
        self.blur_files = sorted(os.listdir(blur_dir))
        self.sharp_files = sorted(os.listdir(sharp_dir))
        self.blur_dir = blur_dir
        self.sharp_dir = sharp_dir
        self.transform = transform
        self.ocr = PaddleOCR(use_angle_cls=True, lang='en', show_log=False)
        self.flip_status = self.detect_flips()
    
    def detect_flips(self):
        flip_status = []
        for blur_file in self.blur_files:
            img_path = os.path.join(self.blur_dir, blur_file)
            img = Image.open(img_path).convert('RGB')
            result = self.ocr.ocr(np.array(img), cls=True)
            flipped = False
            if result and result[0]:
                angle = result[0][0][1][1]  # OCR angle
                flipped = angle > 90 or angle < -90
            flip_status.append(flipped)
        return flip_status
    
    def __len__(self):
        return len(self.blur_files)
    
    def __getitem__(self, idx):
        blur_img = Image.open(os.path.join(self.blur_dir, self.blur_files[idx])).convert('RGB')
        sharp_img = Image.open(os.path.join(self.sharp_dir, self.sharp_files[idx])).convert('RGB')
        if self.flip_status[idx]:
            blur_img = blur_img.transpose(Image.FLIP_LEFT_RIGHT)
            sharp_img = sharp_img.transpose(Image.FLIP_LEFT_RIGHT)
        if self.transform:
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            blur_img = self.transform(blur_img)
            torch.manual_seed(seed)
            sharp_img = self.transform(sharp_img)
        return blur_img, sharp_img

In [ ]:
# Data Preparation
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip
from kornia.augmentation import RandomMotionBlur

image_height, image_width = 128, 256
stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats

train_transform = Compose([
    Resize((image_height, image_width)),
    ToTensor(),
    Normalize(*stats),
    RandomHorizontalFlip(p=0.5),
    RandomMotionBlur(kernel_size=7, angle=0, direction=0.5, p=0.5)  # Horizontal blur
])

valid_transform = Compose([
    Resize((image_height, image_width)),
    ToTensor(),
    Normalize(*stats)
])

# Replace with your dataset paths
blur_dir = "path/to/blur_images"
sharp_dir = "path/to/sharp_images"

dataset = LicensePlateDataset(blur_dir=blur_dir, sharp_dir=sharp_dir, transform=train_transform)

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

# Apply valid_transform to validation and test sets
val_dataset.dataset.transform = valid_transform
test_dataset.dataset.transform = valid_transform

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} pairs, Val: {len(val_dataset)} pairs, Test: {len(test_dataset)} pairs")

In [ ]:
# PaddleOCR Feature Extraction
def extract_ocr_features(images, ocr):
    features = []
    images_np = images.permute(0, 2, 3, 1).cpu().numpy() * 255
    for img in images_np:
        result = ocr.ocr(img.astype(np.uint8), cls=False)
        if result and result[0]:
            text = ''.join([res[1][0] for res in result[0]])
            feat = torch.tensor([ord(c) for c in text[:32]].ljust(32, 0), dtype=torch.float32, device=images.device)
        else:
            feat = torch.zeros(32, dtype=torch.float32, device=images.device)
        features.append(feat)
    return torch.stack(features)

In [ ]:
# Gradient Penalty for WGAN-GP
def compute_gradient_penalty(discriminator, real_samples, fake_samples, device):
    alpha = torch.rand(real_samples.size(0), 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_global, d_patches = discriminator(interpolates)
    fake = torch.ones(d_global.size(), device=device)
    
    gradients = torch.autograd.grad(
        outputs=(d_global, d_patches.mean()),
        inputs=interpolates,
        grad_outputs=(fake, fake),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
# Training Loop with Gradient Accumulation
def train_model(generator, discriminator, train_loader, val_loader, device, num_epochs=50, accum_steps=2):
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))
    perceptual_loss = VGGPerceptualLoss().to(device)
    criterion = nn.L1Loss()
    scaler = torch.cuda.amp.GradScaler()
    ocr = PaddleOCR(use_angle_cls=False, lang='en', show_log=False)
    
    train_losses = {'g_loss': [], 'd_loss': []}
    val_losses = []
    
    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        
        for i, (blur_imgs, sharp_imgs) in enumerate(train_loader):
            blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
            batch_size = blur_imgs.size(0)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            for _ in range(accum_steps):
                with torch.cuda.amp.autocast():
                    real_global, real_patches = discriminator(sharp_imgs)
                    fake_imgs, fake_text = generator(blur_imgs)
                    fake_global_losses, fake_patch_losses = [], []
                    for fake_img in fake_imgs:
                        fake_global, fake_patches = discriminator(fake_img.detach())
                        fake_global_losses.append(torch.mean(fake_global))
                        fake_patch_losses.append(torch.mean(fake_patches))
                    d_loss_real = -torch.mean(real_global) - torch.mean(real_patches)
                    d_loss_fake = sum(fake_global_losses) + sum(fake_patch_losses)
                    gp = compute_gradient_penalty(discriminator, sharp_imgs, fake_imgs[-1], device)
                    d_loss = (d_loss_real + d_loss_fake + 10.0 * gp) / accum_steps
                scaler.scale(d_loss).backward()
            scaler.step(d_optimizer)
            scaler.update()
            
            # Train Generator
            g_optimizer.zero_grad()
            for _ in range(accum_steps):
                with torch.cuda.amp.autocast():
                    fake_imgs, fake_text = generator(blur_imgs)
                    fake_global_losses, fake_patch_losses = [], []
                    for fake_img in fake_imgs:
                        fake_global, fake_patches = discriminator(fake_img)
                        fake_global_losses.append(-torch.mean(fake_global))
                        fake_patch_losses.append(-torch.mean(fake_patches))
                    adv_loss = sum(fake_global_losses) + sum(fake_patch_losses)
                    perc_loss = sum(perceptual_loss(fake_img, sharp_imgs) for fake_img in fake_imgs)
                    with torch.no_grad():
                        gt_text = extract_ocr_features(sharp_imgs, ocr)
                    text_loss = criterion(fake_text, gt_text)
                    g_loss = (adv_loss + 100.0 * perc_loss + 10.0 * text_loss) / accum_steps
                scaler.scale(g_loss).backward()
            scaler.step(g_optimizer)
            scaler.update()
            
            epoch_g_loss += g_loss.item() * accum_steps
            epoch_d_loss += d_loss.item() * accum_steps
            
            if i % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{i}/{len(train_loader)}] "
                      f"D Loss: {d_loss.item():.4f} G Loss: {g_loss.item():.4f}")
        
        # Validation
        generator.eval()
        val_loss = 0.0
        with torch.no_grad():
            for blur_imgs, sharp_imgs in val_loader:
                blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
                with torch.cuda.amp.autocast():
                    fake_imgs, fake_text = generator(blur_imgs)
                    perc_loss = sum(perceptual_loss(fake_img, sharp_imgs) for fake_img in fake_imgs)
                    gt_text = extract_ocr_features(sharp_imgs, ocr)
                    text_loss = criterion(fake_text, gt_text)
                    val_loss += (100.0 * perc_loss + 10.0 * text_loss).item()
        val_loss /= len(val_loader)
        
        train_losses['g_loss'].append(epoch_g_loss / len(train_loader))
        train_losses['d_loss'].append(epoch_d_loss / len(train_loader))
        val_losses.append(val_loss)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] Val Loss: {val_loss:.4f}")
        
        # Save checkpoint
        torch.save(generator.state_dict(), f"generator_epoch_{epoch+1}.pth")
        torch.save(discriminator.state_dict(), f"discriminator_epoch_{epoch+1}.pth")
    
    return train_losses, val_losses

In [ ]:
# Initialize and Train
generator = SwinUNet(img_size=(128, 256)).to(device)
discriminator = Discriminator(img_size=(128, 256)).to(device)

train_losses, val_losses = train_model(generator, discriminator, train_loader, val_loader, device, num_epochs=50)

In [ ]:
# Visualize Training Progress
plt.figure(figsize=(10, 5))
plt.plot(train_losses['g_loss'], label='Generator Loss')
plt.plot(train_losses['d_loss'], label='Discriminator Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Losses')
plt.show()

In [ ]:
# Test and Visualize Results
def visualize_results(generator, test_loader, device, num_samples=5):
    generator.eval()
    with torch.no_grad():
        for i, (blur_imgs, sharp_imgs) in enumerate(test_loader):
            if i >= num_samples:
                break
            blur_imgs, sharp_imgs = blur_imgs.to(device), sharp_imgs.to(device)
            fake_imgs, fake_text = generator(blur_imgs)
            
            # Convert to numpy for visualization
            blur_img = blur_imgs[0].cpu().permute(1, 2, 0).numpy()
            sharp_img = sharp_imgs[0].cpu().permute(1, 2, 0).numpy()
            fake_img = fake_imgs[-1][0].cpu().permute(1, 2, 0).numpy()
            
            plt.figure(figsize=(12, 3))
            plt.subplot(1, 3, 1)
            plt.imshow(blur_img)
            plt.title('Blurred')
            plt.axis('off')
            plt.subplot(1, 3, 2)
            plt.imshow(fake_img)
            plt.title('Deblurred')
            plt.axis('off')
            plt.subplot(1, 3, 3)
            plt.imshow(sharp_img)
            plt.title('Ground Truth')
            plt.axis('off')
            plt.show()

visualize_results(generator, test_loader, device)